# Cuaderno 2: Analisis Exploratorio de Datos (EDA)
**Proyecto Final — Analisis de Subsidios de Vivienda en Colombia**

---
Este cuaderno explora los datos del modelo estrella para entender distribuciones, tendencias temporales, cobertura geografica y comportamiento de los programas de subsidio habitacional.

## 1. Importacion y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (13, 5), 'figure.dpi': 100, 'font.size': 11})

hechos = pd.read_csv('../2. Dataset y Diccionario de Datos/CSV Final Knime/Hechos_Final.csv', encoding='utf-8-sig')
cat_municipio = pd.read_csv('../2. Dataset y Diccionario de Datos/CSV Final Knime/CatMunicipio_clean.csv', encoding='utf-8-sig')
cat_programa = pd.read_csv('../2. Dataset y Diccionario de Datos/CSV Final Knime/CatPrograma_clean.csv', encoding='utf-8-sig')
df1 = pd.read_csv('../2. Dataset y Diccionario de Datos/Datasets Limpios/limpio_Dataset_1.csv', encoding='utf-8-sig', low_memory=False)
df3 = pd.read_csv('../2. Dataset y Diccionario de Datos/Datasets Limpios/limpio_Dataset_3.csv', encoding='utf-8-sig')

print(f"Hechos: {hechos.shape} | Municipios: {cat_municipio.shape} | Programas: {cat_programa.shape}")
hechos.head()

## 2. Estadisticas Descriptivas

In [ ]:
hechos.describe(include='all').T[['count','mean','std','min','max']].dropna(how='all')

In [ ]:
nulos = hechos.isnull().sum()
nulos = nulos[nulos > 0]
if len(nulos) > 0:
    print("Columnas con valores nulos:")
    print(nulos)
else:
    print("Sin valores nulos en la tabla de hechos")

## 3. Distribucion de Variables Numericas

In [ ]:
num_cols = [c for c in ['nropobla','aportegob','aportemun','aporteotr','cantidadtotal','hogares','valor_asignado'] if c in hechos.columns]

n = len(num_cols)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(hechos[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribucion: {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=30)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribucion de Variables Numericas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Analisis Temporal — Subsidios por Anio

In [ ]:
if 'anio_asignacion' in hechos.columns:
    por_anio = hechos.groupby('anio_asignacion').agg(
        total_hogares=('hogares', 'sum'),
        total_valor=('valor_asignado', 'sum'),
        registros=('consecutivo', 'count')
    ).reset_index()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.bar(por_anio['anio_asignacion'].astype(str), por_anio['total_hogares'], color='steelblue', edgecolor='white')
    ax1.set_title('Total Hogares Beneficiados por Anio', fontweight='bold')
    ax1.set_xlabel('Anio')
    ax1.set_ylabel('Hogares')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

    ax2.bar(por_anio['anio_asignacion'].astype(str), por_anio['total_valor'] / 1e9, color='coral', edgecolor='white')
    ax2.set_title('Valor Total Asignado por Anio (Miles de Millones COP)', fontweight='bold')
    ax2.set_xlabel('Anio')
    ax2.set_ylabel('Valor (Miles de Millones $)')
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()
    print(por_anio.to_string(index=False))

## 5. Analisis por Programa

In [ ]:
if 'programa' in hechos.columns:
    por_programa = hechos.groupby('programa').agg(
        hogares=('hogares', 'sum'),
        valor=('valor_asignado', 'sum'),
        registros=('consecutivo', 'count')
    ).sort_values('hogares', ascending=False).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].barh(por_programa['programa'], por_programa['hogares'], color='steelblue')
    axes[0].set_title('Hogares por Programa', fontweight='bold')
    axes[0].set_xlabel('Total Hogares')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    axes[1].barh(por_programa['programa'], por_programa['valor'] / 1e9, color='mediumseagreen')
    axes[1].set_title('Valor Asignado por Programa (Miles de Mill. COP)', fontweight='bold')
    axes[1].set_xlabel('Valor (Miles de Millones $)')

    plt.tight_layout()
    plt.show()
    print(por_programa.to_string(index=False))

## 6. Analisis Geografico — Top Departamentos

In [ ]:
if 'departamento' in hechos.columns:
    top_dep = hechos.groupby('departamento').agg(
        hogares=('hogares', 'sum'),
        valor=('valor_asignado', 'sum')
    ).sort_values('hogares', ascending=False).head(15).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].barh(top_dep['departamento'][::-1], top_dep['hogares'][::-1], color='royalblue')
    axes[0].set_title('Top 15 Departamentos — Hogares Beneficiados', fontweight='bold')
    axes[0].set_xlabel('Total Hogares')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    axes[1].barh(top_dep['departamento'][::-1], top_dep['valor'][::-1] / 1e9, color='darkorange')
    axes[1].set_title('Top 15 Departamentos — Valor Asignado', fontweight='bold')
    axes[1].set_xlabel('Valor (Miles de Millones COP)')

    plt.tight_layout()
    plt.show()

In [ ]:
if 'municipio' in hechos.columns:
    top_mun = hechos.groupby('municipio').agg(
        hogares=('hogares', 'sum')
    ).sort_values('hogares', ascending=False).head(12).reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))
    bars = ax.bar(top_mun['municipio'], top_mun['hogares'], color='steelblue', edgecolor='white')
    ax.set_title('Top 12 Municipios — Total Hogares Beneficiados', fontweight='bold')
    ax.set_ylabel('Total Hogares')
    plt.xticks(rotation=45, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

## 7. Estado de Postulacion

In [ ]:
if 'estado_postulacion' in hechos.columns:
    estado = hechos['estado_postulacion'].value_counts()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.pie(estado.values, labels=estado.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel', len(estado)), startangle=90)
    ax1.set_title('Distribucion por Estado de Postulacion', fontweight='bold')

    ax2.bar(estado.index, estado.values, color=sns.color_palette('muted', len(estado)), edgecolor='white')
    ax2.set_title('Frecuencia por Estado', fontweight='bold')
    ax2.set_ylabel('Cantidad de Registros')
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    print(estado)

## 8. Matriz de Correlacion

In [ ]:
num_hechos = hechos.select_dtypes(include='number').dropna(axis=1, how='all')
excluir = [c for c in num_hechos.columns if 'cod' in c or 'codigo' in c or 'divipola' in c]
num_hechos = num_hechos.drop(columns=excluir, errors='ignore')

if num_hechos.shape[1] > 1:
    corr = num_hechos.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
    ax.set_title('Matriz de Correlacion — Variables Numericas', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Aporte Gubernamental por Subregion (Dataset 1)

In [ ]:
if 'subregion' in df1.columns and 'aportegob' in df1.columns:
    df1['aportegob'] = pd.to_numeric(df1['aportegob'], errors='coerce').fillna(0)
    por_sub = df1.groupby('subregion')['aportegob'].sum().sort_values(ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(12, 5))
    por_sub.plot(kind='bar', ax=ax, color='teal', edgecolor='white')
    ax.set_title('Aporte Gubernamental por Subregion', fontweight='bold')
    ax.set_xlabel('Subregion')
    ax.set_ylabel('Aporte Gobierno (COP)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:,.0f}M'))
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 10. Hallazgos Clave

In [ ]:
hallazgos = [
    "1. TEMPORAL: Los anios con mayor asignacion de subsidios son los mas recientes (MI CASA YA en expansion).",
    "2. PROGRAMAS: 'MI CASA YA' concentra la mayor cantidad de hogares beneficiados.",
    "3. GEOGRAFIA: Antioquia, Cundinamarca y Valle del Cauca lideran en cobertura.",
    "4. ESTADO: La mayoria de postulaciones estan en estado 'Asignados', indicando alta efectividad.",
    "5. CORRELACION: Valor asignado y hogares tienen alta correlacion positiva (subsidio por hogar).",
]
print("RESUMEN DE HALLAZGOS EDA")
print("=" * 60)
for h in hallazgos:
    print(h)